# Population variability notebook

This notebook computes two per-synapse metrics from the single-trial `image_identity` responses:

1. **Within-image variance**: mean trial-to-trial variance across repeated presentations of the same image  
2. **Between-image variance**: variance of the per-image mean responses across images

It follows the same session/asset loading style as your FVE notebook and merges the output back onto the activation table so each row keeps `session_id`, `session_type`, `dmd`, `synapse_id`, `response_class`, and the DMD-derived `depth`.


In [ ]:

import os
import glob
import warnings
import numpy as np
import pandas as pd
import itertools
from pathlib import Path
from PNW_cmap import PNW_cmap
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

sns.set_style('white')
params = {
    'legend.fontsize': 'x-large',
    'axes.labelsize': 'xx-large',
    'axes.titlesize': 'xx-large',
    'xtick.labelsize': 'xx-large',
    'ytick.labelsize': 'xx-large'
}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))


In [ ]:

%load_ext autoreload
%autoreload 2
%matplotlib notebook


In [ ]:
savepath = r'C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\figures'

In [ ]:

target_mice = [
    803496,
    804730, 804733, 810196,
    809047, 803121,
    826033, 838410, 834788,
]

registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check", "volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Loaded {len(assets)} session assets")


In [ ]:

st_paths = [
    glob.glob(os.path.join(asset.derived_dir, '**', 'glutamate_single_trial_df.npz'), recursive=True)[0]
    for asset in assets
]

act_summary_path = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\data\activation_summary.csv"
act_summary = pd.read_csv(act_summary_path)

# one row per synapse in your activation table; keep all activation metadata
act_summary = act_summary.drop(columns=[c for c in act_summary.columns if c.startswith('Unnamed:')], errors='ignore')
activation_synapses = act_summary.drop_duplicates(['session_id', 'dmd', 'synapse_id']).copy()

display(activation_synapses.head())
print(f"{len(activation_synapses)} unique synapses in activation table")


In [ ]:

def _safe_first(glob_result):
    return glob_result[0] if len(glob_result) else np.nan


asset_rows = []
for (_, row), asset, st_path in zip(process_df.iterrows(), assets, st_paths):
    meta = getattr(asset, 'metadata', {}) or {}
    session_id = getattr(asset, 'session_id', row.get('session_id', Path(getattr(asset, 'session_dir', '')).name))

    session_type = row.get('session_type', np.nan)
    if pd.isna(session_type):
        session_type = getattr(asset, 'session_type', meta.get('session_type', np.nan))

    subject_id = row.get('subject_id', np.nan)
    if pd.isna(subject_id):
        subject_id = getattr(asset, 'subject_id', meta.get('subject_id', np.nan))

    asset_rows.append({
        'session_id': session_id,
        'subject_id': subject_id,
        'session_type': session_type,
        'session_dir': getattr(asset, 'session_dir', np.nan),
        'derived_dir': getattr(asset, 'derived_dir', np.nan),
        'st_path': st_path,
        'dmd1_depth': meta.get('dmd1_depth', np.nan),
        'dmd2_depth': meta.get('dmd2_depth', np.nan),
    })

asset_table = pd.DataFrame(asset_rows).drop_duplicates('session_id').copy()
display(asset_table.head())


In [ ]:

# Merge session_type onto the synapse table now, before computing variability
activation_synapses = activation_synapses.merge(
    asset_table[['session_id', 'session_type', 'dmd1_depth', 'dmd2_depth']],
    on='session_id',
    how='left',
)

activation_synapses['depth'] = np.where(
    activation_synapses['dmd'].eq('DMD1'),
    activation_synapses['dmd1_depth'],
    activation_synapses['dmd2_depth'],
)

display(activation_synapses.head())


## Variability settings

By default this uses a **baseline-subtracted post-stimulus mean** as the trial amplitude:

- baseline window: `0:50`
- response window: `50:100`

At 200 Hz this is:
- baseline: `0.00–0.25 s`
- response: `0.25–0.50 s`

If you want strict parity with the earlier FVE notebook, set `baseline_slice = None` and it will use the raw post-stimulus mean only.


In [ ]:

response_slice = (50, 100)
baseline_slice = (0, 50)   # set to None for post-only amplitudes
min_trials_per_image = 2   # minimum valid repeated trials required for a given image


In [ ]:

def nanmean_no_warning(x, axis):
    x = np.asarray(x, dtype=float)
    count = np.sum(~np.isnan(x), axis=axis)
    total = np.nansum(x, axis=axis)

    out = np.full(np.shape(total), np.nan, dtype=float)
    np.divide(total, count, out=out, where=count > 0)
    return out


def nanvar_no_warning(x, axis, ddof=0):
    x = np.asarray(x, dtype=float)
    mean = nanmean_no_warning(x, axis=axis)
    mean_expanded = np.expand_dims(mean, axis=axis)

    sq = (x - mean_expanded) ** 2
    count = np.sum(~np.isnan(x), axis=axis)
    total = np.nansum(sq, axis=axis)

    denom = count - ddof
    out = np.full(np.shape(total), np.nan, dtype=float)
    np.divide(total, denom, out=out, where=denom > 0)
    return out


def compute_trial_amplitudes(traces, response_slice=(50, 100), baseline_slice=(0, 50)):
    """
    Parameters
    ----------
    traces : array, shape (n_trials, n_synapses, n_samples)
    """
    post = nanmean_no_warning(traces[..., response_slice[0]:response_slice[1]], axis=-1)

    if baseline_slice is None:
        return post

    pre = nanmean_no_warning(traces[..., baseline_slice[0]:baseline_slice[1]], axis=-1)
    return post #- pre


In [ ]:

def compute_session_variability_from_st(
    st_path,
    activation_synapses_session,
    response_slice=(50, 100),
    baseline_slice=(0, 50),
    min_trials_per_image=2,
):
    """
    Returns one row per synapse with within-image and between-image variance.
    """
    payload = np.load(st_path, allow_pickle=True)['data'][0]
    session_id = payload['metadata']['session_id']

    out_rows = []

    for dmd in ('DMD1', 'DMD2'):
        if dmd not in payload:
            continue

        dmd_payload = payload[dmd]
        synapse_ids = np.asarray(dmd_payload['synapse_ids']).astype(str)
        image_dict = dmd_payload['image_identity']

        # Restrict to synapses that exist in the activation table for this session+dmd
        syn_meta = activation_synapses_session.query("session_id == @session_id and dmd == @dmd").copy()
        if syn_meta.empty:
            continue

        keep_mask = np.isin(synapse_ids, syn_meta['synapse_id'].values)
        if not np.any(keep_mask):
            continue

        kept_synapse_ids = synapse_ids[keep_mask]
        kept_indices = np.where(keep_mask)[0]

        image_names = list(image_dict.keys())
        n_images = len(image_names)
        n_syn = len(kept_synapse_ids)

        image_means = np.full((n_images, n_syn), np.nan, dtype=float)
        image_vars = np.full((n_images, n_syn), np.nan, dtype=float)
        image_counts = np.zeros((n_images, n_syn), dtype=int)

        pooled_trial_values = [[] for _ in range(n_syn)]

        for i, image_name in enumerate(image_names):
            traces = np.asarray(image_dict[image_name], dtype=float)[:, kept_indices, :]
            amps = compute_trial_amplitudes(
                traces,
                response_slice=response_slice,
                baseline_slice=baseline_slice,
            )  # shape (n_trials, n_syn)

            counts = np.sum(~np.isnan(amps), axis=0)
            means = nanmean_no_warning(amps, axis=0)
            variances = nanvar_no_warning(amps, axis=0, ddof=1)

            image_means[i] = means
            image_vars[i] = np.where(counts >= min_trials_per_image, variances, np.nan)
            image_counts[i] = counts

            for j in range(n_syn):
                vals = amps[:, j]
                vals = vals[~np.isnan(vals)]
                if len(vals):
                    pooled_trial_values[j].append(vals)

        within_image_var = nanmean_no_warning(image_vars, axis=0)

        between_image_var = np.full(n_syn, np.nan, dtype=float)
        n_images_used = np.sum(~np.isnan(image_vars), axis=0)
        mean_trials_per_image = nanmean_no_warning(
            np.where(image_counts >= min_trials_per_image, image_counts, np.nan),
            axis=0,
        )

        total_trial_var = np.full(n_syn, np.nan, dtype=float)

        for j in range(n_syn):
            vals = image_means[:, j]
            vals = vals[~np.isnan(vals)]
            if len(vals) >= 2:
                between_image_var[j] = np.var(vals, ddof=1)

            if len(pooled_trial_values[j]):
                pooled = np.concatenate(pooled_trial_values[j])
                if len(pooled) >= 2:
                    total_trial_var[j] = np.var(pooled, ddof=1)

        syn_results = pd.DataFrame({
            'session_id': session_id,
            'dmd': dmd,
            'synapse_id': kept_synapse_ids,
            'within_image_var': within_image_var,
            'between_image_var': between_image_var,
            'total_trial_var': total_trial_var,
            'n_images_used': n_images_used,
            'mean_trials_per_image': mean_trials_per_image,
        })

        syn_results['within_frac_total'] = syn_results['within_image_var'] / syn_results['total_trial_var']
        syn_results['within_to_between_ratio'] = syn_results['within_image_var'] / syn_results['between_image_var']

        syn_results = syn_results.merge(
            syn_meta.drop(columns=['dmd1_depth', 'dmd2_depth'], errors='ignore'),
            on=['session_id', 'dmd', 'synapse_id'],
            how='left',
        )

        out_rows.append(syn_results)

    if out_rows:
        return pd.concat(out_rows, ignore_index=True)

    return pd.DataFrame()


In [ ]:

all_var = []

for _, row in asset_table.iterrows():
    st_path = row['st_path']
    session_id = row['session_id']
    print(session_id)

    if pd.isna(st_path) or not os.path.exists(st_path):
        print(f"  missing single-trial file: {st_path}")
        continue

    session_var = compute_session_variability_from_st(
        st_path=st_path,
        activation_synapses_session=activation_synapses,
        response_slice=response_slice,
        baseline_slice=baseline_slice,
        min_trials_per_image=min_trials_per_image,
    )

    if session_var.empty:
        print("  no rows returned")
        continue

    all_var.append(session_var)

variance_df = pd.concat(all_var, ignore_index=True) if all_var else pd.DataFrame()
display(variance_df.head())
print(variance_df.shape)


In [ ]:

# Clean up columns and set plotting helpers
variance_df = variance_df.copy()

variance_df = variance_df[
    ~variance_df['depth'].isna()
].copy()

for col in ['within_image_var', 'between_image_var', 'total_trial_var', 'within_frac_total', 'within_to_between_ratio']:
    variance_df[f'log10_{col}'] = np.where(variance_df[col] > 0, np.log10(variance_df[col]), np.nan)

depth_order = sorted(variance_df['depth'].dropna().unique())
variance_df['depth'] = pd.Categorical(variance_df['depth'], categories=depth_order, ordered=True)

display(
    variance_df.groupby('depth')[['within_image_var', 'between_image_var', 'within_frac_total']]
    .agg(['count', 'median', 'mean'])
)


In [ ]:

cl, cmap, cp = PNW_cmap.get_PNW_cmap('Sailboat', n_colors=max(len(depth_order), 1))
palette = cp[::-1][:len(depth_order)]

fig, ax = plt.subplots(figsize=(4.2, 4.8))

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

for spine in ['left', 'right', 'top', 'bottom']:
    ax.spines[spine].set_linewidth(2)

sns.despine()

plot_df = variance_df.dropna(subset=['log10_within_image_var']).copy()

sns.stripplot(
    data=plot_df,
    x='depth',
    y='log10_within_image_var',
    order=depth_order,
    palette=palette,
    size=2.5,
    alpha=0.8,
    ax=ax,
)

medians = (
    plot_df.groupby('depth', observed=False)['log10_within_image_var']
    .median()
    .reindex(depth_order)
    .values
)
ax.plot(range(len(depth_order)), medians, color='k', lw=3, marker='o', zorder=20)

ax.set_xlabel('Depth from pia (μm)')
ax.set_ylabel('log10 variance')
ax.set_title('Trial variance')
ax.set_ylim(-1,8)
fig.tight_layout()

filen = 'within_image_variance_by_depth'
save_figure(fig, os.path.join(savepath, filen), formats=['.pdf', '.png'], dpi=300)


In [ ]:
# ------------------------------------------------------------
# Pairwise depth comparisons
# ------------------------------------------------------------
depth_order = [25, 100, 200, 250]

pairwise_rows = []

for d1, d2 in itertools.combinations(depth_order, 2):
    x = plot_df.loc[plot_df["depth"] == d1, "log10_within_image_var"].dropna().to_numpy()
    y = plot_df.loc[plot_df["depth"] == d2, "log10_within_image_var"].dropna().to_numpy()

    if len(x) == 0 or len(y) == 0:
        continue

    stat, p = mannwhitneyu(x, y, alternative="two-sided")

    pairwise_rows.append(
        {
            "depth_1": d1,
            "depth_2": d2,
            "n_1": len(x),
            "n_2": len(y),
            "median_1": np.median(x),
            "median_2": np.median(y),
            "mean_1": np.mean(x),
            "mean_2": np.mean(y),
            "mw_u": stat,
            "p_uncorrected": p,
        }
    )

pairwise_stats = pd.DataFrame(pairwise_rows)

# ------------------------------------------------------------
# Multiple-comparisons correction
# ------------------------------------------------------------
reject, p_fdr, _, _ = multipletests(
    pairwise_stats["p_uncorrected"].values,
    alpha=0.05,
    method="fdr_bh",
)

pairwise_stats["p_fdr_bh"] = p_fdr
pairwise_stats["significant_fdr_bh"] = reject

pairwise_stats = pairwise_stats.sort_values("p_fdr_bh").reset_index(drop=True)

pairwise_stats

In [ ]:

fig, ax = plt.subplots(figsize=(4.2, 4.8))

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

for spine in ['left', 'right', 'top', 'bottom']:
    ax.spines[spine].set_linewidth(2)

sns.despine()

plot_df = variance_df.dropna(subset=['log10_between_image_var']).copy()

sns.stripplot(
    data=plot_df,
    x='depth',
    y='log10_between_image_var',
    order=depth_order,
    palette=palette,
    size=2.5,
    alpha=0.8,
    ax=ax,
)

medians = (
    plot_df.groupby('depth', observed=False)['log10_between_image_var']
    .median()
    .reindex(depth_order)
    .values
)
ax.plot(range(len(depth_order)), medians, color='k', lw=3, marker='o', zorder=20)

ax.set_xlabel('Depth from pia (μm)')
ax.set_ylabel('log10 variance')
ax.set_title('Between-image mean variance')
ax.set_ylim(-4,10)
fig.tight_layout()

filen = 'between_image_variance_by_depth'
save_figure(fig, os.path.join(savepath, filen), formats=['.pdf', '.png'], dpi=300)


In [ ]:
# ------------------------------------------------------------
# Pairwise depth comparisons
# ------------------------------------------------------------
depth_order = [25, 100, 200, 250]

pairwise_rows = []

for d1, d2 in itertools.combinations(depth_order, 2):
    x = plot_df.loc[plot_df["depth"] == d1, "log10_between_image_var"].dropna().to_numpy()
    y = plot_df.loc[plot_df["depth"] == d2, "log10_between_image_var"].dropna().to_numpy()

    if len(x) == 0 or len(y) == 0:
        continue

    stat, p = mannwhitneyu(x, y, alternative="two-sided")

    pairwise_rows.append(
        {
            "depth_1": d1,
            "depth_2": d2,
            "n_1": len(x),
            "n_2": len(y),
            "median_1": np.median(x),
            "median_2": np.median(y),
            "mean_1": np.mean(x),
            "mean_2": np.mean(y),
            "mw_u": stat,
            "p_uncorrected": p,
        }
    )

pairwise_stats = pd.DataFrame(pairwise_rows)

# ------------------------------------------------------------
# Multiple-comparisons correction
# ------------------------------------------------------------
reject, p_fdr, _, _ = multipletests(
    pairwise_stats["p_uncorrected"].values,
    alpha=0.05,
    method="fdr_bh",
)

pairwise_stats["p_fdr_bh"] = p_fdr
pairwise_stats["significant_fdr_bh"] = reject

pairwise_stats = pairwise_stats.sort_values("p_fdr_bh").reset_index(drop=True)

pairwise_stats

In [ ]:
plot_df[plot_df['log10_within_image_var']>3]

In [ ]:

# Optional: save the per-synapse table for later use in plotting or stats
out_csv = r"C:\Users\andrew.shelton\Downloads\synapse_variability_summary.csv"
variance_df.to_csv(out_csv, index=False)
print(out_csv)
